# 🚀 OpenHands AI Agent trên Google Colab

Notebook này cho phép bạn chạy **OpenHands AI Agent** trực tiếp trên Google Colab kết hợp với mô hình **Gemini 3 Flash / 3.8 Flash**.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tuna1710/openhands-workspace/blob/main/openhands_colab.ipynb)

> **Lưu ý quan trọng về môi trường Google Colab:**
> Máy ảo Google Colab chạy trong container giới hạn không hỗ trợ Docker daemon. Vì vậy, notebook này cấu hình OpenHands chạy ở chế độ **Native Process / CLI & Web TUI Mode** (không cần Docker), giúp Agent thao tác trực tiếp với hệ điều hành và file trên Colab với tốc độ nhanh nhất mà không gặp lỗi 502 Bad Gateway.

## 📦 Bước 1: Cài đặt OpenHands & Cloudflared

In [ ]:
# 1. Cài đặt OpenHands CLI & Agent SDK
!pip install -q openhands openhands-ai nest_asyncio

# 2. Tải Cloudflared để mở cổng Web UI ra Internet (hoàn toàn miễn phí, không cần tài khoản)
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb > /dev/null 2>&1
print('✅ Cài đặt thành công OpenHands & Cloudflared!')

## 🔑 Bước 2: Nhập API Key & Thiết lập Model Gemini

In [ ]:
import os
import json
from pathlib import Path
from getpass import getpass

# Nhập Gemini API Key từ Google AI Studio (https://aistudio.google.com/)
api_key = getpass('🔑 Nhập Google Gemini API Key của bạn: ')

# Thiết lập biến môi trường cho OpenHands & LiteLLM
os.environ['GEMINI_API_KEY'] = api_key
os.environ['LLM_API_KEY'] = api_key
os.environ['LLM_MODEL'] = 'gemini/gemini-3.8-flash'
os.environ['OPENHANDS_SUPPRESS_BANNER'] = '1'
os.environ['PYTHONWARNINGS'] = 'ignore'
os.environ['RUNTIME'] = 'process'

# Lưu sẵn cấu hình vào ~/.openhands/agent_settings.json để không bị hỏi wizard
openhands_dir = Path.home() / '.openhands'
openhands_dir.mkdir(parents=True, exist_ok=True)
config_data = {
    'llm': {
        'model': 'gemini/gemini-3.8-flash',
        'api_key': api_key
    }
}
with open(openhands_dir / 'agent_settings.json', 'w') as f:
    json.dump(config_data, f, indent=2)
with open(openhands_dir / 'settings.json', 'w') as f:
    json.dump(config_data, f, indent=2)

print('✅ Đã cấu hình Model: gemini/gemini-3.8-flash')


## 🎯 Cách 1: Giao việc trực tiếp cho Agent ngay trong Colab (Khuyên dùng - Nhanh & Ổn định nhất)
Thực thi trực tiếp trong cell Colab, tự động tạo file và chạy lệnh mà không cần qua tunnel hay trình duyệt phụ.

In [ ]:
# Đổi nội dung task theo ý muốn của bạn
task = '''Hãy tạo một ứng dụng web Game rắn săn mồi (Snake Game) với HTML, CSS và JavaScript, sau đó viết file README.md giải thích code.'''

# Ghi nội dung task vào task.txt (tránh lỗi ký tự đặc biệt, dấu ngoặc kép hoặc ngắt dòng trên shell)
with open('task.txt', 'w', encoding='utf-8') as f:
    f.write(task.strip())

# Chạy Agent trong Colab với chế độ headless & tự động phê duyệt lệnh qua file task.txt
!openhands --headless --override-with-envs --always-approve -f task.txt


## 🌐 Cách 2: Khởi chạy Giao diện Web (OpenHands Web UI) với Cloudflare Tunnel
Chạy ô bên dưới để khởi động Web App TUI của OpenHands và lấy đường dẫn Cloudflare Tunnel để thao tác trên trình duyệt.

In [ ]:
import subprocess
import time
import re

# 1. Khởi chạy OpenHands Web (native process, không dùng Docker)
server_process = subprocess.Popen(
    ['openhands', 'web', '--host', '0.0.0.0', '--port', '3000'],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)
print('⏳ Đang khởi động OpenHands web server trên port 3000...')
time.sleep(5)

# Kiểm tra nếu tiến trình thoát sớm
if server_process.poll() is not None:
    print('❌ Lỗi khi khởi động OpenHands server:')
    print(server_process.stdout.read())
else:
    # 2. Mở đường hầm Cloudflare Tunnel công khai
    tunnel_process = subprocess.Popen(
        ['cloudflared', 'tunnel', '--url', 'http://127.0.0.1:3000'],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True
    )

    print('🔗 Đang tạo link truy cập Web UI...')
    for line in tunnel_process.stdout:
        match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
        if match:
            public_url = match.group(0)
            print(f'\n🎉 BẤM VÀO ĐÂY ĐỂ MỞ OPENHANDS WEB UI:\n👉 {public_url}\n')
            print('⚡ Lưu ý: Giữ ô này đang chạy (running) để duy trì kết nối.')
            break